In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install fasttext

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 1.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-2.13.6-py3-none-any.whl.metadata (9.5 kB)
Using cached pybind11-2.13.6-py3-none-any.whl (243 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp310-cp310-linux_x86_64.whl size=4296183 sha256=95b4153ea15fa2d100016725936066a501404589a53dee5512232d002fea7e6b
  Stored in directory: /root/.cache/pip/wheels/0d/a2/00/81db54d3e6a8199b829d58e02cec2ddb20ce3e59fad8d3c92a
Successfully built fasttext


In [3]:
cd drive/MyDrive/SNN_Aspect_Term_Extraction/SemEval_Data

/content/drive/MyDrive/SNN_Aspect_Term_Extraction/SemEval_Data


In [4]:
cd SemEval/

/content/drive/MyDrive/SNN_Aspect_Term_Extraction/SemEval_Data/SemEval


In [ ]:
!ls

accessories_aspect_terms_v1.txt  glove.840B.300d.txt  SemEval16
accessories_reviews_v1.txt	 SemEval14	      yelp_aspect_terms_v2.txt
cc.en.300.bin			 SemEval15	      yelp_reviews_v2.txt


In [5]:
import os, spacy
import numpy as np
import fasttext
import xml.etree.ElementTree as ET

In [ ]:
# Load the small English language model
nlp = spacy.load("en_core_web_sm")

In [ ]:
def find_aspect_term_indices(text, ats):
    indices = {}
    for at in ats:
        start_idx = text.find(at)
        if start_idx != -1:
            end_idx = start_idx + len(at)
            indices[at] = (start_idx, end_idx)
        else:
            indices[at] = None  # If the subtext is not found
    return indices

In [ ]:
def getData(path, typeNum, augmentedType, datasetType):

  reviews = []
  tokenized_reviews = []
  aspect_terms = []
  bio_format = []

  # Parse the XML file
  tree = ET.parse(path)

  # Get the root element
  root = tree.getroot()


  if datasetType == "Train":
    print(datasetType)

    if augmentedType == "accessories":
      with open('accessories_reviews_v1.txt', 'r') as f:
        accessories_reviews = f.readlines()

      with open('accessories_aspect_terms_v1.txt', 'r') as f:
          accessories_aspect_terms = f.readlines()

      augment_reviews = [x.strip() for x in accessories_reviews]
      augment_aspect_terms = [x.strip() for x in accessories_aspect_terms]

    elif augmentedType == "yelp":
      with open('yelp_reviews_v2.txt', 'r') as f:
        yelp_reviews = f.readlines()

      with open('yelp_aspect_terms_v2.txt', 'r') as f:
          yelp_aspect_terms = f.readlines()

      augment_reviews = [x.strip() for x in yelp_reviews]
      augment_aspect_terms = [x.strip() for x in yelp_aspect_terms]

    else:
      pass


    for text, ats in zip(augment_reviews, augment_aspect_terms):
        ats = ats.lower().split(';') # Aspect terms
        text = text.lower().strip()
        tokenized_text = [token.text for token in nlp(text)]

        try:
          # Find indices of each subtext
          indices = find_aspect_term_indices(text, ats)
          aspect_term = []
          bio = ["O"]*len(tokenized_text)

          for at, idx_pair in indices.items():
              at_from, at_to = idx_pair

              idx = len([token.text for token in nlp(text[:at_from])])
              idx1 = len([token.text for token in nlp(text[:at_to])])

              i = 0
              for index in range(idx,idx1):
                if (i==0):
                  bio[index] = "B"
                  i = 1
                elif (i==1):
                  bio[index] = "I"

              aspect_term.append(at)

          aspect_terms.append(aspect_term)
          bio_format.append(bio)
          reviews.append(text)
          tokenized_reviews.append(tokenized_text)
        except:
          continue


  if (typeNum == 15) or (typeNum == 16):
    # Loop through each review and sentence
    for review in root.findall('Review'):
        review_id = review.get('rid')
        print(f"Review ID: {review_id}")

        for sentence in review.findall('.//sentence'):
            sentence_id = sentence.get('id')
            text = sentence.find('text').text
            print(f"  Sentence ID: {sentence_id}")
            print(f"    Text: {text}")

            text = text.lower().strip()
            reviews.append(text)
            tokenized_text = [token.text for token in nlp(text)]
            tokenized_reviews.append(tokenized_text)

            # Fetch opinions
            opinions = sentence.find('Opinions')
            aspect_term = []
            bio = ["O"]*len(tokenized_text)

            if opinions is not None:
                for opinion in opinions.findall('Opinion'):
                    target = opinion.get('target').lower()
                    if target == "null":
                      continue
                    category = opinion.get('category')
                    polarity = opinion.get('polarity')
                    opinion_from = eval(opinion.get('from'))
                    opinion_to = eval(opinion.get('to'))
                    print(f"    Opinion:")
                    print(f"      Target: {target}")
                    print(f"      Category: {category}")
                    print(f"      Polarity: {polarity}")
                    print(f"      From: {opinion_from}")
                    print(f"      To: {opinion_to}")

                    idx = len([token.text for token in nlp(text[:opinion_from])])
                    idx1 = len([token.text for token in nlp(text[:opinion_to])])
                    i = 0
                    for index in range(idx,idx1):
                      if (i==0):
                        bio[index] = "B"
                        i = 1
                      elif (i==1):
                        bio[index] = "I"

                    aspect_term.append(target)

                aspect_terms.append(aspect_term)
                bio_format.append(bio)

            else:
                aspect_terms.append(aspect_term)
                bio_format.append(bio)

            print("\n")


  else:
    for sentence in root.findall('.//sentence'):
      sentence_id = sentence.get('id')
      text = sentence.find('text').text
      print(f"  Sentence ID: {sentence_id}")
      print(f"    Text: {text}")

      text = text.lower().strip()
      reviews.append(text)
      tokenized_text = [token.text for token in nlp(text)]
      tokenized_reviews.append(tokenized_text)

      # Fetch opinions
      opinions = sentence.find('aspectTerms')
      aspect_term = []
      bio = ["O"]*len(tokenized_text)
      if opinions is not None:
          for opinion in opinions.findall('aspectTerm'):
              target = opinion.get('term').lower()
              if target == "null":
                continue
              polarity = opinion.get('polarity')
              opinion_from = eval(opinion.get('from'))
              opinion_to = eval(opinion.get('to'))
              print(f"    Opinion:")
              print(f"      Target: {target}")
              print(f"      Polarity: {polarity}")
              print(f"      From: {opinion_from}")
              print(f"      To: {opinion_to}")

              idx = len([token.text for token in nlp(text[:opinion_from])])
              idx1 = len([token.text for token in nlp(text[:opinion_to])])
              i = 0
              for index in range(idx,idx1):
                if (i==0):
                  bio[index] = "B"
                  i = 1
                elif (i==1):
                  bio[index] = "I"

              aspect_term.append(target.lower())

          aspect_terms.append(aspect_term)
          bio_format.append(bio)

      else:
          aspect_terms.append(aspect_term)
          bio_format.append(bio)

      print("\n")


  # Save each info
  path1 = path.split("/")[0]
  name = path.split("/")[-1].split(".")[0]

  # Save the dictionary as an npz file
  np.savez(path1+'/'+name+'_augment.npz', {"reviews":reviews, "tokenized_reviews":tokenized_reviews, "aspect_terms":aspect_terms, "bio_format":bio_format})

  return reviews, tokenized_reviews, aspect_terms, bio_format

In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval14/Laptops_Train.xml', typeNum = 14, augmentedType = 'accessories', datasetType = 'Train')

Streaming output truncated to the last 5000 lines.
    Opinion:
      Target: windows vista system
      Polarity: neutral
      From: 20
      To: 40


  Sentence ID: 1533
    Text: She said its very user friendly.


  Sentence ID: 2950
    Text: Kind of annoying, but I still love the laptop.


  Sentence ID: 3057
    Text: cosmetically, the only thing they changed was 2 of the Function keys at the top.
    Opinion:
      Target: function keys
      Polarity: neutral
      From: 55
      To: 68


  Sentence ID: 1536
    Text: I am very happy I bought this Mac, well worth the extra money.


  Sentence ID: 1001
    Text: But sadly the replacement froze-up while updating the BIOS again and shut down and would not turn back on.
    Opinion:
      Target: bios
      Polarity: negative
      From: 54
      To: 58


  Sentence ID: 830
    Text:   There's literally no way to make it sing with Vista.
    Opinion:
      Target: vista
      Polarity: negative
      From: 48
      To: 53


  Sent

In [ ]:
print(reviews[0])
print(tokenized_reviews[0])
print(aspect_terms[0])
print(bio_format[0])

finally, i can eat and text without leaving fingerprints behind!!!!!!!!
['finally', ',', 'i', 'can', 'eat', 'and', 'text', 'without', 'leaving', 'fingerprints', 'behind', '!', '!', '!', '!', '!', '!', '!', '!']
['fingerprints']
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'B', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval14/Laptops_Test_Data_phaseB.xml', typeNum = 14, augmentedType = '', datasetType = 'Test')

Streaming output truncated to the last 5000 lines.
      Target: built-in applications
      Polarity: None
      From: 38
      To: 59
    Opinion:
      Target: iphoto
      Polarity: None
      From: 65
      To: 71


  Sentence ID: 29:358
    Text: This Mac Mini makes the Macbook Pro seem slow.


  Sentence ID: 895:1
    Text: I did swap out the hard drive for a Samsung 830 SSD which I highly recommend.
    Opinion:
      Target: hard drive
      Polarity: None
      From: 19
      To: 29
    Opinion:
      Target: samsung 830 ssd
      Polarity: None
      From: 36
      To: 51


  Sentence ID: 836:1
    Text: I bought this MacBook Pro to replace my six-year-old PC (a Sony Vaio), but it was basically no better than my old PC, so I returned it.


  Sentence ID: 1087:1
    Text: I wanted a simple, reliable  laptop.


  Sentence ID: 846:2
    Text: Cheaper than buying it @ apple too!


  Sentence ID: 1063:165
    Text: I have just had to learn to be a little harder typer than on my l

In [ ]:
print(reviews[0])
print(tokenized_reviews[0])
print(aspect_terms[0])
print(bio_format[0])

boot time is super fast, around anywhere from 35 seconds to 1 minute.
['boot', 'time', 'is', 'super', 'fast', ',', 'around', 'anywhere', 'from', '35', 'seconds', 'to', '1', 'minute', '.']
['boot time']
['B', 'I', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval15/ABSA-15_Restaurants_Train_Final.xml', typeNum = 15, augmentedType = 'yelp', datasetType = 'Train')

Streaming output truncated to the last 5000 lines.
  Sentence ID: 1746113:0
    Text: We arrived on time for our reservation and seated promptly.


  Sentence ID: 1746113:1
    Text: The menu has so many fish items and oysters.
    Opinion:
      Target: menu
      Category: FOOD#STYLE_OPTIONS
      Polarity: positive
      From: 4
      To: 8


  Sentence ID: 1746113:2
    Text: We all ordered different entrees so we could share.


  Sentence ID: 1746113:3
    Text: The fish was really,really fresh.
    Opinion:
      Target: fish
      Category: FOOD#QUALITY
      Polarity: positive
      From: 4
      To: 8


  Sentence ID: 1746113:4
    Text: I lived in Maine for ten years and grew up on fish.


  Sentence ID: 1746113:5
    Text: We all agreed that mare is one of the best seafood restaurants in New York.
    Opinion:
      Target: mare
      Category: RESTAURANT#GENERAL
      Polarity: positive
      From: 19
      To: 23


Review ID: 1748813
  Sentence ID: 1748813:0
    Text: I st

In [ ]:
print(reviews[0])
print(tokenized_reviews[0])
print(aspect_terms[0])
print(bio_format[0])

watching the saints game and having some chargrilled oysters
['watching', 'the', 'saints', 'game', 'and', 'having', 'some', 'chargrilled', 'oysters']
['saints game', 'chargrilled oysters']
['O', 'O', 'B', 'I', 'O', 'O', 'O', 'B', 'I']


In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval15/ABSA15_Restaurants_Test.xml', typeNum = 15, augmentedType = '', datasetType = 'Test')

Streaming output truncated to the last 5000 lines.
    Opinion:
      Target: quacamole
      Category: FOOD#QUALITY
      Polarity: positive
      From: 0
      To: 9
    Opinion:
      Target: wings with chimmichuri
      Category: FOOD#QUALITY
      Polarity: positive
      From: 43
      To: 65


  Sentence ID: P#9:6
    Text: A weakness is the chicken in the salads.
    Opinion:
      Target: chicken in the salads
      Category: FOOD#QUALITY
      Polarity: negative
      From: 18
      To: 39


  Sentence ID: P#9:7
    Text: It's just average, just shredded, no seasoning on it.


  Sentence ID: P#9:8
    Text: Also, I personally wasn't a fan of the portobello and asparagus mole.
    Opinion:
      Target: portobello and asparagus mole
      Category: FOOD#QUALITY
      Polarity: negative
      From: 39
      To: 68


  Sentence ID: P#9:9
    Text: Overall, decent food at a good price, with friendly people.
    Opinion:
      Target: food
      Category: FOOD#QUALITY
      Polari

In [ ]:
print(reviews[0])
print(tokenized_reviews[0])
print(aspect_terms[0])
print(bio_format[0])

love al di la
['love', 'al', 'di', 'la']
['al di la']
['O', 'B', 'I', 'I']


In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval16/ABSA16_Restaurants_Train_SB1_v2.xml', typeNum = 16, augmentedType = 'yelp', datasetType = 'Train')

Streaming output truncated to the last 5000 lines.
      Category: FOOD#QUALITY
      Polarity: positive
      From: 4
      To: 22


  Sentence ID: P#9:4
    Text: the spinach is fresh, definately not frozen...
    Opinion:
      Target: spinach
      Category: FOOD#QUALITY
      Polarity: positive
      From: 4
      To: 11


  Sentence ID: P#9:5
    Text: quacamole at pacifico is yummy, as are the wings with chimmichuri.
    Opinion:
      Target: quacamole
      Category: FOOD#QUALITY
      Polarity: positive
      From: 0
      To: 9
    Opinion:
      Target: wings with chimmichuri
      Category: FOOD#QUALITY
      Polarity: positive
      From: 43
      To: 65


  Sentence ID: P#9:6
    Text: A weakness is the chicken in the salads.
    Opinion:
      Target: chicken in the salads
      Category: FOOD#QUALITY
      Polarity: negative
      From: 18
      To: 39


  Sentence ID: P#9:7
    Text: It's just average, just shredded, no seasoning on it.


  Sentence ID: P#9:8
    Text

(7593, 7593, 7593, 7593)

In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval16/EN_REST_SB1_TEST.xml.gold', typeNum = 16, augmentedType = '', datasetType = 'Test')

Streaming output truncated to the last 5000 lines.
    Text: i am never disappointed with there food.
    Opinion:
      Target: food
      Category: FOOD#QUALITY
      Polarity: positive
      From: 35
      To: 39


  Sentence ID: en_OpenSesame_477970770:4
    Text: the atmosphere is great.
    Opinion:
      Target: atmosphere
      Category: AMBIENCE#GENERAL
      Polarity: positive
      From: 4
      To: 14


  Sentence ID: en_OpenSesame_477970770:5
    Text: and all you other people that have a problem get some help


Review ID: en_MercedesRestaurant_478010601
  Sentence ID: en_MercedesRestaurant_478010601:0
    Text: great lunch spot
    Opinion:
      Target: lunch spot
      Category: RESTAURANT#GENERAL
      Polarity: positive
      From: 6
      To: 16


  Sentence ID: en_MercedesRestaurant_478010601:1
    Text: – Great financial district mexican spot.
    Opinion:
      Target: mexican spot
      Category: RESTAURANT#GENERAL
      Polarity: positive
      From: 27
      To

(676, 676, 676, 676)

In [ ]:
print(reviews[0])
print(tokenized_reviews[0])
print(aspect_terms[0])
print(bio_format[0])

yum!
['yum', '!']
[]
['O', 'O']


In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval14/Restaurants_Train.xml', typeNum = 14, augmentedType = 'yelp', datasetType = 'Train')

Streaming output truncated to the last 5000 lines.
    Text: Warm, comfortable surroundings, nice appointments (witness the etched glass and brickwork separating the dining rooms).
    Opinion:
      Target: surroundings
      Polarity: positive
      From: 18
      To: 30
    Opinion:
      Target: dining rooms
      Polarity: neutral
      From: 105
      To: 117


  Sentence ID: 2029
    Text: We ran a little late for the reservation but it wasn't a problem to get our table immediately.
    Opinion:
      Target: reservation
      Polarity: neutral
      From: 29
      To: 40
    Opinion:
      Target: table
      Polarity: positive
      From: 76
      To: 81


  Sentence ID: 2258
    Text: Kudos Haru


  Sentence ID: 2341
    Text: We arrived on time for our reservation and seated promptly.The
    Opinion:
      Target: reservation
      Polarity: positive
      From: 27
      To: 38


  Sentence ID: 679
    Text: I plan on eating here more often.


  Sentence ID: 2004
    Text: I

(8637, 8637, 8637, 8637)

In [ ]:
reviews, tokenized_reviews, aspect_terms, bio_format = getData(path = 'SemEval14/Restaurants_Test_Data_phaseB.xml', typeNum = 14, augmentedType = '', datasetType = 'Test')

Streaming output truncated to the last 5000 lines.
      From: 4
      To: 13


  Sentence ID: 11351324#0#2
    Text: The menu, which changes seasonally, shows both regional and international influences.
    Opinion:
      Target: menu
      Polarity: None
      From: 4
      To: 8


  Sentence ID: 11624981#549489#5
    Text: but their mac cheese was YUMMY!
    Opinion:
      Target: mac cheese
      Polarity: None
      From: 10
      To: 20


  Sentence ID: 11624981#549489#6
    Text: their brunch menu had something for everyone.
    Opinion:
      Target: brunch menu
      Polarity: None
      From: 6
      To: 17


  Sentence ID: 11624981#549489#7
    Text: jazz singer had a nice voice + she made us all get up to dance to shake some cals to eat some more.
    Opinion:
      Target: jazz singer
      Polarity: None
      From: 0
      To: 11


  Sentence ID: 15069510#648179#0
    Text: I often find myself with time to kill in Times Square, which is a shame since I don't love this ar

(800, 800, 800, 800)

In [ ]:
print(reviews[0])
print(tokenized_reviews[0])
print(aspect_terms[0])
print(bio_format[0])

the bread is top notch as well.
['the', 'bread', 'is', 'top', 'notch', 'as', 'well', '.']
['bread']
['O', 'B', 'O', 'O', 'O', 'O', 'O', 'O']


# Word2idx

In [ ]:
def word2idx_and_idx2word(path, path1):
  word2idx = {}
  idx2word = {}
  word2idx["<pad>"] = 0
  idx2word[0] = "<pad>"

  for tk_review in np.load(path, allow_pickle=True)['arr_0'].tolist()['tokenized_reviews'] + np.load(path1, allow_pickle=True)['arr_0'].tolist()['tokenized_reviews']:
    for tk in tk_review:
      if tk not in word2idx:
        idx = len(word2idx)
        word2idx[tk] = idx
        idx2word[idx] = tk

  # Save each info
  path1 = path.split("/")[0]
  name = "Laptops" if "Laptops" in path else "Restaurants"

  # Save the dictionary as an npz file
  np.savez(path1+'/'+name+"_word2idx_idx2word"+'_augment.npz', {"word2idx":word2idx, "idx2word":idx2word})

  return word2idx, idx2word

In [ ]:
path="SemEval14/Laptops_Train_augment.npz"
path1="SemEval14/Laptops_Test_Data_phaseB_augment.npz"
word2idx, idx2word = word2idx_and_idx2word(path, path1)

In [ ]:
path="SemEval15/ABSA-15_Restaurants_Train_Final_augment.npz"
path1="SemEval15/ABSA15_Restaurants_Test_augment.npz"
word2idx, idx2word = word2idx_and_idx2word(path, path1)

In [ ]:
path="SemEval16/ABSA16_Restaurants_Train_SB1_v2_augment.npz"
path1="SemEval16/EN_REST_SB1_TEST_augment.npz"
word2idx, idx2word = word2idx_and_idx2word(path, path1)

In [ ]:
path="SemEval14/Restaurants_Train_augment.npz"
path1="SemEval14/Restaurants_Test_Data_phaseB_augment.npz"
word2idx, idx2word = word2idx_and_idx2word(path, path1)

In [ ]:
# Load the pre-trained FastText model
fastText_model = fasttext.load_model("cc.en.300.bin")

def getEmbedding(embedding_module_path, word2idx_path, dim=300, type_pre_embedding=''):

    path1 = word2idx_path.split("/")[0]
    name = "Laptops" if "Laptops" in word2idx_path else "Restaurants"
    embedding_save_path = path1+'/'+name+'_'+type_pre_embedding+'_embedding'+'_augment.npy'

    # Load the saved npz file
    loaded_dict = np.load(word2idx_path, allow_pickle=True)['arr_0'].tolist()
    word2idx = loaded_dict['word2idx']

    embeddings = np.zeros((len(word2idx), dim))

    if type_pre_embedding == 'glove':
      words_in_vocab = []
      with open(embedding_module_path) as f:
        for l in f:
            w_emd_pair = l.rstrip().split(' ')
            word = w_emd_pair[0]
            embedding = np.array([float(e) for e in w_emd_pair[1:]])
            if word in word2idx:
                words_in_vocab.append(word)
                embeddings[word2idx[word]] = embedding

      # OOV words have also embeddings
      for word in word2idx:
        if word not in words_in_vocab:
          if (word == "<pad>"):
            continue
          else:
            embeddings[word2idx[word]] = fastText_model.get_word_vector(word)


    elif type_pre_embedding == 'word2vec':
      import gensim.downloader as api
      # Load the pre-trained Word2Vec model
      word2vec_model = api.load(embedding_module_path)
      for word in word2idx:
        if word in word2vec_model.key_to_index:
            embedding = word2vec_model[word]
            embeddings[word2idx[word]] = embedding
        else:
            # OOV words have also embeddings
            if (word == "<pad>"):
              continue
            else:
              embeddings[word2idx[word]] = fastText_model.get_word_vector(word)


    elif type_pre_embedding == 'fasttext':
      # Unhash below coden when to train
      # loaded_dict1 = np.load(embedding_module_path, allow_pickle=True)['arr_0'].tolist()
      # from gensim.models import FastText
      # # Training FastText on domain-specific data
      # fastText_model = FastText(vector_size=dim, window=3, min_count=1, sentences=loaded_dict1['tokenized_reviews'], epochs=10)
      for word in word2idx:
        if word in fastText_model.words:
            embedding = fastText_model.get_word_vector(word)
            embeddings[word2idx[word]] = embedding
        else:
            # OOV words have also embeddings
            if (word == "<pad>"):
              continue
            else:
              embeddings[word2idx[word]] = fastText_model.get_word_vector(word)


    # Saving OOV (Out of vocabulary) words
    with open(path1+'/'+name+'_'+type_pre_embedding+"_oov_augment.txt", "w") as fw:
        for w in word2idx :
            if embeddings[word2idx[w]].sum()==0.:
                fw.write(w+"\n")

    # Saving embedding file
    np.save(embedding_save_path, embeddings.astype('float32') )


In [ ]:
getEmbedding(embedding_module_path="glove.840B.300d.txt", word2idx_path="SemEval14/Laptops_word2idx_idx2word_augment.npz", dim=300, type_pre_embedding='glove')

In [ ]:
getEmbedding(embedding_module_path="glove.840B.300d.txt", word2idx_path="SemEval14/Restaurants_word2idx_idx2word_augment.npz", dim=300, type_pre_embedding='glove')

In [ ]:
getEmbedding(embedding_module_path="glove.840B.300d.txt", word2idx_path="SemEval15/Restaurants_word2idx_idx2word_augment.npz", dim=300, type_pre_embedding='glove')

In [ ]:
getEmbedding(embedding_module_path="glove.840B.300d.txt", word2idx_path="SemEval16/Restaurants_word2idx_idx2word_augment.npz", dim=300, type_pre_embedding='glove')